# 06 - Collecte multi-ligues & Re-analyse

On elargit a toutes les competitions StatsBomb disponibles.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsbombpy import sb
plt.rcParams.update({"font.size": 12, "axes.titlesize": 14, "axes.titleweight": "bold",
                      "figure.figsize": (14, 6), "figure.dpi": 100,
                      "axes.spines.top": False, "axes.spines.right": False})
print("Setup OK")

---
## 1. Lister les competitions disponibles

In [ ]:
comps = sb.competitions()
print("Competitions disponibles :")
print()
for _, row in comps.groupby(["competition_id", "competition_name"]).agg(n_seasons=("season_id", "count")).reset_index().sort_values("n_seasons", ascending=False).iterrows():
    cid = row["competition_id"]
    name = row["competition_name"]
    ns = row["n_seasons"]
    print(f"  {name:<35s} (id={cid:<5}) {ns} saison(s)")

---
## 2. Selection des competitions

On prend toutes les ligues domestiques + Champions League + tournois majeurs.

In [ ]:
# ADAPTER les IDs si necessaire (verifier avec la cellule precedente)
TARGET_COMPS = [
    (11, "La Liga"),
    (2, "Premier League"),
    (12, "Serie A"),
    (9, "Bundesliga"),
    (7, "Ligue 1"),
    (16, "Champions League"),
    (87, "Copa del Rey"),
    (43, "FIFA World Cup"),
    (55, "UEFA Euro"),
]

all_targets = []
for comp_id, comp_name in TARGET_COMPS:
    seasons = comps[comps["competition_id"] == comp_id]
    for _, s in seasons.iterrows():
        all_targets.append({"competition_id": comp_id, "season_id": s["season_id"],
                            "competition_name": comp_name, "season_name": s["season_name"]})

targets_df = pd.DataFrame(all_targets)
n_total = len(targets_df)
print(f"Plan de collecte : {n_total} competition/saisons")
print()
for comp_name, group in targets_df.groupby("competition_name"):
    seasons_list = ", ".join(group["season_name"].tolist())
    print(f"  {comp_name}: {len(group)} saisons")

---
## 3. Collecte massive

15-30 min selon le nombre de competitions. Le cache evite les re-telechargements.

In [ ]:
from src.collect import StatsBombCollector
from src.clean import ShotDataCleaner

collector = StatsBombCollector()
cleaner = ShotDataCleaner()

all_shots = []
collection_log = []

for _, target in targets_df.iterrows():
    comp_id = target["competition_id"]
    season_id = target["season_id"]
    comp_name = target["competition_name"]
    season_name = target["season_name"]
    
    print(f"--- {comp_name} {season_name} ---")
    try:
        shots = collector.collect_season(comp_id, season_id)
        if shots is not None and len(shots) > 0:
            shots["league"] = comp_name
            all_shots.append(shots)
            n = len(shots)
            nm = shots["match_id"].nunique()
            print(f"    OK : {n} tirs, {nm} matchs")
            collection_log.append({"league": comp_name, "season": season_name, "n_shots": n, "n_matches": nm, "status": "OK"})
        else:
            print(f"    Vide")
            collection_log.append({"league": comp_name, "season": season_name, "n_shots": 0, "n_matches": 0, "status": "VIDE"})
    except Exception as e:
        emsg = str(e)[:50]
        print(f"    ERREUR : {emsg}")
        collection_log.append({"league": comp_name, "season": season_name, "n_shots": 0, "n_matches": 0, "status": emsg})

log_df = pd.DataFrame(collection_log)
print("
" + "=" * 60)
print("BILAN DE COLLECTE")
print("=" * 60)
for league, grp in log_df.groupby("league"):
    ok = grp[grp["status"] == "OK"]
    total_shots = ok["n_shots"].sum()
    total_matches = ok["n_matches"].sum()
    print(f"  {league:<25s} {len(ok)}/{len(grp)} saisons  {total_shots:>6} tirs  {total_matches:>4} matchs")

In [ ]:
# Fusionner
if all_shots:
    shots_all_raw = pd.concat(all_shots, ignore_index=True)
    n_total = len(shots_all_raw)
    n_matchs = shots_all_raw["match_id"].nunique()
    print(f"Dataset brut total : {n_total} tirs, {n_matchs} matchs")
    print()
    for league, grp in shots_all_raw.groupby("league"):
        nm = grp["match_id"].nunique()
        print(f"  {league:<25s} {len(grp):>6} tirs, {nm:>4} matchs")
else:
    print("ERREUR : aucun tir collecte !")

---
## 4. Nettoyage

In [ ]:
shots_clean_all = cleaner.clean(shots_all_raw, source="statsbomb")
cleaner.print_quality_report()
shots_clean_all.to_parquet("../data/processed/shots_clean_all.parquet", index=False)
n_clean = len(shots_clean_all)
print(f"
Sauvegarde : {n_clean} tirs propres")

---
## 5. Feature engineering complet

In [ ]:
from src.features import UnderperformanceFeatures
feat = UnderperformanceFeatures()

# Timelines (peut prendre quelques minutes)
timeline_all = feat.build_all_timelines(shots_clean_all)
feat.print_summary(timeline_all)
timeline_all.to_parquet("../data/processed/timelines_all.parquet", index=False)
n_tl = len(timeline_all)
print(f"
Timelines sauvegardees : {n_tl} observations")

In [ ]:
episodes_all = feat.create_underperformance_episodes(timeline_all)
episodes_all.to_parquet("../data/processed/episodes_all.parquet", index=False)
n_ep = len(episodes_all)
print(f"Episodes : {n_ep}")

---
## 6. Re-analyse complete

In [ ]:
from src.analysis import WindowAnalysis
wa = WindowAnalysis()
tl_all = feat.filter_complete_windows(timeline_all, window=10)
n_obs = len(tl_all)
n_m = tl_all["match_id"].nunique()
print(f"Dataset filtre : {n_obs} observations, {n_m} matchs")

baseline = wa.compute_baseline(tl_all, window=10)
p_base = baseline["p_concede"]
print(f"Baseline P(conceder 10min) : {p_base:.4f}")

results_05 = wa.event_study(tl_all, threshold=0.5, window=10)
wa.print_event_study(results_05, label="DATASET COMPLET - Seuil 0.5")

results_03 = wa.event_study(tl_all, threshold=0.3, window=10)
wa.print_event_study(results_03, label="DATASET COMPLET - Seuil 0.3")

In [ ]:
# Stratification
print("STRATIFICATION PAR SCORE STATE (dataset complet, seuil 0.5)")
print("=" * 65)
strat = wa.stratified_analysis(tl_all, threshold=0.5, window=10)
for state, res in strat.items():
    wa.print_event_study(res, label="Score state : " + state.upper())

In [ ]:
# Permutation
perm = wa.permutation_test(tl_all, threshold=0.5, window=10, n_permutations=10000)
wa.print_permutation(perm)

---
## 7. Forest plot par ligue

In [ ]:
# Effet par ligue
match_league = shots_clean_all[["match_id", "league"]].drop_duplicates()
tl_with_league = tl_all.merge(match_league, on="match_id", how="left")

forest_data = []
for league in sorted(tl_with_league["league"].dropna().unique()):
    subset = tl_with_league[tl_with_league["league"] == league]
    n_treated = len(subset[subset["cum_underperf"] >= 0.5])
    n_control = len(subset[subset["cum_underperf"] < 0.5])
    if n_treated < 30 or n_control < 30:
        continue
    res = wa.event_study(subset, threshold=0.5, window=10)
    if res is None:
        continue
    diff = res["concede"]["difference"]
    pval = res["concede"]["p_value"]
    n_m = subset["match_id"].nunique()
    forest_data.append({"league": league, "diff": diff, "p_value": pval,
                        "n_treated": n_treated, "n_control": n_control, "n_matches": n_m})

forest_df = pd.DataFrame(forest_data)
print(forest_df.to_string(index=False))

In [ ]:
# Forest plot
BG = "#FAFAFA"
RED = "#E63946"
BLUE = "#457B9D"
DARK = "#1D3557"
GRAY = "#ADB5BD"

if len(forest_df) > 0:
    forest_df = forest_df.sort_values("diff")
    fig, ax = plt.subplots(figsize=(12, max(4, len(forest_df) * 0.8)))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(BG)
    
    y_pos = range(len(forest_df))
    colors = [RED if p < 0.05 else GRAY for p in forest_df["p_value"]]
    ax.barh(y_pos, forest_df["diff"] * 100, color=colors, alpha=0.7, edgecolor="white", height=0.6)
    ax.axvline(0, color=DARK, linewidth=1, linestyle="--")
    
    labels = []
    for _, row in forest_df.iterrows():
        labels.append(row["league"] + " (" + str(row["n_matches"]) + " matchs)")
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=11)
    
    for i, (_, row) in enumerate(forest_df.iterrows()):
        d = row["diff"] * 100
        p = row["p_value"]
        sig = " *" if p < 0.05 else ""
        offset = 0.3 if d >= 0 else -0.3
        ha = "left" if d >= 0 else "right"
        ax.text(d + offset, i, str(round(d, 1)) + "%" + sig, va="center", ha=ha, fontsize=10, fontweight="bold")
    
    ax.set_xlabel("Delta P(conceder 10min) en %", fontsize=12)
    ax.set_title("Effet de l underperformance par ligue", fontsize=15, fontweight="bold", pad=15)
    ax.text(0.98, 0.02, "Rouge = significatif (p < 0.05)
Gris = non significatif",
            transform=ax.transAxes, ha="right", va="bottom", fontsize=9, color=GRAY)
    plt.tight_layout()
    plt.savefig("../outputs/figures/pub_07_forest_plot.png", dpi=200, bbox_inches="tight", facecolor=BG)
    plt.show()
else:
    print("Pas assez de ligues pour un forest plot")

---
## 8. Null model sur dataset complet

In [ ]:
from src.models import NullModelSimulation
null = NullModelSimulation()
obs_effect = results_05["concede"]["difference"]
print(f"Effet observe : {obs_effect:+.5f}")
print("Null model 500 sims (peut prendre 15-30 min)...")

null_all = null.run(shots_df=shots_clean_all, observed_effect=obs_effect,
                    threshold=0.5, window=10, n_simulations=500, seed=42)
null.print_results(null_all)

---
## 9. Resume complet

In [ ]:
print("
" + "=" * 70)
print("  RESULTATS MULTI-LIGUES")
print("=" * 70)

n_shots_t = len(shots_clean_all)
n_matchs_t = shots_clean_all["match_id"].nunique()
n_leagues = shots_clean_all["league"].nunique() if "league" in shots_clean_all.columns else 0
print(f"
  Dataset : {n_shots_t} tirs, {n_matchs_t} matchs, {n_leagues} ligues")

diff05 = results_05["concede"]["difference"]
pval05 = results_05["concede"]["p_value"]
print(f"
  Event study seuil 0.5 : Delta = {diff05:+.4f}, p = {pval05:.4f}")

print("
  Stratification :")
for state, res in strat.items():
    if res.get("skipped"):
        print(f"    {state:10s} : SKIP")
    else:
        d = res["concede"]["difference"]
        p = res["concede"]["p_value"]
        sig = "*" if p < 0.05 else ""
        print(f"    {state:10s} : Delta = {d:+.4f}, p = {p:.4f} {sig}")

pp = perm["p_value"]
print(f"
  Permutation : p = {pp:.4f}")

nobs = null_all["observed_effect"]
nmean = null_all["null_mean"]
np_v = null_all["p_value"]
print(f"
  Null model : observe={nobs:+.4f}, null={nmean:+.4f}, p={np_v:.4f}")

if len(forest_df) > 0:
    n_neg = len(forest_df[forest_df["diff"] < 0])
    print(f"
  Forest : {n_neg}/{len(forest_df)} ligues avec effet negatif")

print("
" + "=" * 70)
ci_lo = null_all["ci_95"][0]
ci_hi = null_all["ci_95"][1]
if nobs < ci_lo:
    print("  CONCLUSION : Effet PLUS NEGATIF que le null -> Confounding")
elif ci_lo <= nobs <= ci_hi:
    print("  CONCLUSION : Effet DANS le null -> Pas d effet reel")
else:
    print("  CONCLUSION : Effet PLUS POSITIF -> Vrai effet")
print("=" * 70)